# Audit the citation-aware MUFASA training set

This notebook runs the same bundle-aware support router audited in
`sample-training-set.ipynb`, but over the complete frozen corpus split.

The builder preserves strict outputs and three parallel mixed training lanes:

- `sft_examples.parquet`: verified FACTUAL/REASONING SFT;
- `sft_mixed.parquet`: strict SFT plus support-unverified, enriched SFT;
- `dpo_pairs.parquet`: verified PREFERENCE data;
- `preference_mixed.parquet`: all structurally valid preference pairs;
- `reranker_mixed.parquet`: all structurally valid reranker pairs;
- unresolved support and all rerankers remain visible in `quarantine.parquet`;
- structurally invalid/conflicting duplicate rows -> discarded audit.

**Production is deliberately paused.** The default `AUDIT_ONLY` mode renders
author-year/provenance targets for inspection but is mechanically forbidden
from publishing. Citation metadata is diagnostic and filters zero pairs.
Switch to `TRAINING` and `PUBLISH=True` only after reviewing the separate
raw-versus-document citation audit below.

Production outputs are immutable Parquet generations under `training_set/runs/`.
`training_set/LATEST.json` is switched only after every Parquet has been
written and verified. No refusal examples are manufactured.
The final cell displays counts and deterministic, text-truncated samples for
all seven outputs; truncation affects only the notebook display, never Parquet.


In [14]:
# =============================== controls ==================================
import importlib
import os
import sys
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

try:
    from google.colab import drive
except ImportError:
    drive = None
else:
    drive.mount("/content/drive")

# On Colab, edit this one path only if your MUFASA folder is elsewhere.
DRIVE_DATA = Path("/content/drive/MyDrive/MUFASA/01-data-engineering/data-extraction")
override = os.environ.get("MUFASA_DATA_DIR", "").strip()
candidates = ([Path(override)] if override else []) + ([DRIVE_DATA] if drive else [])
candidates += [
    folder for start in [Path.cwd(), *Path.cwd().parents]
    for folder in (start, start / "01-data-engineering" / "data-extraction")
]
DATA = next((folder.resolve() for folder in candidates
             if (folder / "mufasa_corpus" / "parsed" / "markdown").is_dir()), None)
if DATA is None:
    raise FileNotFoundError("Set DRIVE_DATA or MUFASA_DATA_DIR to the data-extraction folder")
for module in ("mufasa_dataset.py", "mufasa_citations.py", "mufasa_training_builder.py"):
    if not (DATA / module).is_file():
        raise FileNotFoundError(f"Place {module} beside this notebook at {DATA}")
sys.path.insert(0, str(DATA))

import mufasa_citations as citations
import mufasa_dataset as funnel
import mufasa_training_builder as builder
importlib.reload(citations)
importlib.reload(funnel)
importlib.reload(builder)

# None means the complete corpus. Set a small integer only for a deliberate
# dry run; a limited run has a different immutable run identity.

# PAPER_LIMIT = 10
# PROVENANCE_MODE = "AUDIT_ONLY"
# PUBLISH = False

PAPER_LIMIT = None
PROVENANCE_MODE = "TRAINING"
PUBLISH = True

CITATION_METADATA_PATH = DATA / "citation_metadata.parquet"
ROUTER_WORKERS = min(12, max(1, (os.cpu_count() or 2) - 1))

CONFIG = builder.BuildConfig(
    extraction_root=DATA / "extraction_output",
    markdown_dir=DATA / "mufasa_corpus" / "parsed" / "markdown",
    split_manifest=DATA / "corpus_splits" / "manifest.parquet",
    output_root=DATA / "training_set",
    seed=7,
    router_workers=ROUTER_WORKERS,
    max_evidence_spans=3,
    max_evidence_chars=8_000,
    open_share=0.45,
    closed_share=0.45,
    dual_share=0.10,
    paper_limit=PAPER_LIMIT,
    progress_every=100,
    citation_metadata_path=CITATION_METADATA_PATH,
    provenance_mode=PROVENANCE_MODE,
)

frozen = builder.load_frozen_manifest(CONFIG.split_manifest)
print("data root      :", DATA)
print("papers by split:", frozen.split.value_counts().to_dict())
print("router workers :", ROUTER_WORKERS)
print("paper limit    :", PAPER_LIMIT or "FULL CORPUS")
print("provenance mode:", PROVENANCE_MODE)
print("publish        :", PUBLISH)
print("PRODUCTION PAUSED: AUDIT_ONLY cannot publish.")


data root      : C:\CodingWorld\Hackathons\AfricanDeepTechChallenge\MUFASA\01-data-engineering\data-extraction
papers by split: {'train': 9806, 'evaluate': 337, 'test': 337}
router workers : 7
paper limit    : FULL CORPUS
provenance mode: TRAINING
publish        : True
PRODUCTION PAUSED: AUDIT_ONLY cannot publish.


In [15]:
# ===================== citation metadata (audit only) ======================
import hashlib

table_dir, _ = builder.resolve_extraction_tables(CONFIG.extraction_root)
pair_papers = pd.read_parquet(
    table_dir / "training_pairs.parquet", columns=["paper_id"]
)["paper_id"].map(funnel.clean).drop_duplicates().tolist()
selected_citation_papers = sorted(
    pair_papers,
    key=lambda paper: hashlib.sha256(f"{CONFIG.seed}:{paper}".encode()).hexdigest(),
)
if PAPER_LIMIT is not None:
    selected_citation_papers = selected_citation_papers[:PAPER_LIMIT]

CITATION_AUDIT = citations.prepare_citation_metadata(
    split_manifest=CONFIG.split_manifest,
    documents_path=DATA / "mufasa_corpus" / "manifests" / "documents.parquet",
    authors_cache_path=DATA / "production" / "authors_cache.parquet",
    markdown_root=CONFIG.markdown_dir,
    paper_ids=selected_citation_papers,
    output_path=CITATION_METADATA_PATH,
)

print("CITATION AUDIT SUMMARY — descriptive only; zero papers/pairs filtered")
display(pd.DataFrame([citations.audit_summary(CITATION_AUDIT)]))
audit_columns = [
    "paper_id", "openalex_label", "citation_label",
    "openalex_year", "document_year", "author_status",
    "year_status", "citation_status", "metadata_source",
    "collision_count", "fallback_used",
]
display(CITATION_AUDIT[audit_columns].sample(
    min(10, len(CITATION_AUDIT)), random_state=7,
).reset_index(drop=True))
print("citation table:", CITATION_METADATA_PATH)
print("Production remains paused. Review raw OpenAlex vs selected citation labels above.")


citation audit (9,720 papers):   0%|          | 0/9720 [00:00<?, ?paper/s]

CITATION AUDIT SUMMARY — descriptive only; zero papers/pairs filtered


,rows,citation_status,author_status,year_status,fallback_rows,ambiguous_author_year_rows,audit_mode
0,9720,"{'VERIFIED_DOCUMENT': 4508, 'METADATA_ONLY': 2...","{'VERIFIED_DOCUMENT': 7959, 'CORRECTED_DOCUMEN...","{'VERIFIED_DOCUMENT': 5564, 'METADATA_ONLY': 2...",0,2452,AUDIT_ONLY


,paper_id,openalex_label,citation_label,openalex_year,document_year,author_status,year_status,citation_status,metadata_source,collision_count,fallback_used
0,W2955842313,"Adeniyi et al., 2019","Adeniyi et al., 2019",2019,2019,VERIFIED_DOCUMENT,VERIFIED_DOCUMENT,VERIFIED_DOCUMENT,OPENALEX+DOCUMENT,1,False
1,W4416820871,"Effiong et al., 2025","Effiong et al., 2025",2025,,VERIFIED_DOCUMENT,METADATA_ONLY,METADATA_ONLY,OPENALEX+DOCUMENT,1,False
2,W1494606931,"Oboh et al., 2007","Oboh et al., 2007",2007,2007,VERIFIED_DOCUMENT,VERIFIED_DOCUMENT,VERIFIED_DOCUMENT,OPENALEX+DOCUMENT,1,False
3,W2130917951,"Modrek et al., 2014","Modrek et al., 2014",2014,2014,VERIFIED_DOCUMENT,VERIFIED_DOCUMENT,VERIFIED_DOCUMENT,OPENALEX+DOCUMENT,1,False
4,W4409524385,"Helen & OKOH, 2025","Helen & OKOH, 2025",2025,2025,VERIFIED_DOCUMENT,VERIFIED_DOCUMENT,VERIFIED_DOCUMENT,OPENALEX+DOCUMENT,2,False
5,W4403257704,"Haftu et al., 2024","Cheepurupalli et al., 2024",2024,,CORRECTED_DOCUMENT,METADATA_ONLY,CORRECTED_DOCUMENT,OPENALEX+DOCUMENT_CORRECTION,1,False
6,W2947363430,"Ugbomeh & Diboyesuku, 2019","Ugbomeh & Diboyesuku, 2019",2019,2019,VERIFIED_DOCUMENT,VERIFIED_DOCUMENT,VERIFIED_DOCUMENT,OPENALEX+DOCUMENT,1,False
7,W4220899593,"Adeyemo et al., 2022","Adeyemo et al., 2022",2022,,VERIFIED_DOCUMENT,METADATA_ONLY,METADATA_ONLY,OPENALEX+DOCUMENT,2,False
8,W1950094360,"Folorunso, 2013","Folorunso, 2013",2013,,VERIFIED_DOCUMENT,METADATA_ONLY,METADATA_ONLY,OPENALEX+DOCUMENT,3,False
9,W2034333264,"Iwalokun & Iwalokun, 2007","Iwalokun & Iwalokun, 2007",2007,2007,VERIFIED_DOCUMENT,VERIFIED_DOCUMENT,VERIFIED_DOCUMENT,OPENALEX+DOCUMENT,1,False


citation table: C:\CodingWorld\Hackathons\AfricanDeepTechChallenge\MUFASA\01-data-engineering\data-extraction\citation_metadata.parquet
Production remains paused. Review raw OpenAlex vs selected citation labels above.


In [ ]:
# ======================== route, assemble, publish ==========================
OUTCOME = builder.build_training_set(CONFIG, publish=PUBLISH)

print("\nFUNNEL")
display(pd.DataFrame(OUTCOME.stages))

print("\nOUTPUT ROWS")
for name, frame in OUTCOME.frames.items():
    print(f"  {name:<18} {len(frame):>10,}")

sft = OUTCOME.frames["sft_examples"]
sft_mixed = OUTCOME.frames["sft_mixed"]
dpo = OUTCOME.frames["dpo_pairs"]
preference_mixed = OUTCOME.frames["preference_mixed"]
reranker_mixed = OUTCOME.frames["reranker_mixed"]
if len(sft):
    print("\nSFT split/mode")
    display(pd.crosstab(sft["split"], sft["mode"], margins=True))
    print("SFT support routes:", sft.support_route.value_counts().to_dict())
    print("estimated token mass by assignment:",
          OUTCOME.manifest["curriculum_assignment_token_mass"])
if len(dpo):
    print("DPO by split:", dpo.split.value_counts().to_dict())
if len(sft_mixed):
    print("SFT mixed tiers:", sft_mixed.verification_tier.value_counts().to_dict())
if len(preference_mixed):
    print("Preference mixed tiers:", preference_mixed.verification_tier.value_counts().to_dict())
if len(reranker_mixed):
    print("Reranker mixed rows:", len(reranker_mixed))

print("\nrun id :", OUTCOME.run_id)
print("run dir:", OUTCOME.run_dir or "not published")


0 input pairs                      kept   479,143  removed        0  9,720 papers
0b citation provenance             kept     9,720  removed        0  mode=TRAINING; audit only, zero pair filters; status={'CORRECTED_DOCUMENT': 1118, 'VERIFIED_DOCUMENT': 4508, 'CONFLICT': 1850, 'METADATA_ONLY': 2244}
1 evidence bundles                 kept   440,971  removed   38,172  
2 structural gates                 kept   462,812  removed   16,331  {'placeholder ellipsis in target': 96, 'judged not africa relevant': 750, 'task was truncated': 4965, 'chosen and rejected are identical': 214, 'positive and hard negative are identical': 82, 'blank question': 3092, 'blank answer': 2941, 'blank positive quote': 2443, 'blank hard negative quote': 2688, 'blank chosen': 1346, 'blank rejected': 1211, 'wrong language': 1814, 'judged not real science': 801, 'text encoding corruption': 53}
3 duplicate safety                 kept   462,425  removed      387  
   routed 100/9,462 papers in 10.4 min
   routed 200/

In [ ]:
# ================= citation-aware target samples (full text) ==============
from IPython.display import Markdown, display

preview = OUTCOME.frames["sft_mixed"].sample(
    min(5, len(OUTCOME.frames["sft_mixed"])), random_state=7,
).reset_index(drop=True)
for number, row in preview.iterrows():
    display(Markdown(
        f"### Sample {number + 1} — {row['mode']} / {row['verification_tier']}\n\n"
        f"**Question**\n\n{row['question']}\n\n"
        f"**Assistant target**\n\n```text\n{row['response']}\n```\n\n"
        f"Raw OpenAlex: `{row['citation_raw_label']}`  \n"
        f"Selected citation: `{row['citation_label']}`  \n"
        f"Status/source: `{row['citation_status']}` / `{row['citation_metadata_source']}`"
    ))
print("These are audit previews only; no Parquet generation was published.")


### Sample 1 — CLOSED / VERIFIED

**Question**

What solvent and volume were used to elute the sample extracts from the clean-up column?

**Assistant target**

```text
20 mL of n hexane.

Provenance: LEARNED_STUDY
Citation: (Ouabo et al., 2018)
Study basis: discipline: ENVIRONMENTAL_SCIENCE; population: Surface soil samples (0–20 cm depth); location: Douala, Cameroun; period: between February and June 2017; focus: Soils from informal e-waste recycling sites in Douala, Cameroun
```

Raw OpenAlex: `Ouabo et al., 2018`  
Selected citation: `Ouabo et al., 2018`  
Status/source: `VERIFIED_DOCUMENT` / `OPENALEX+DOCUMENT`

### Sample 2 — OPEN / UNVERIFIED

**Question**

Which GenBank accession numbers were assigned to the Nigerian sequences?

**Assistant target**

```text
MK509014 (NGR_CAstV_Ch1), MK509015 (NGR_CAstV_Ch2), MK518374 (NGR_CAstV_Ch3), MK518375 (NGR_CAstV_Ch4) for Nigerian CAstV; MN026334 (NGR_ARV_Ch) for Nigerian ARV; MN026333 (NGR_ChPV_Ch) for Nigerian ChPV and MN026335 (NGR_ANV_Ch) for Nigerian ANV.

Provenance: UNVERIFIED_STUDY
Citation: (Adebiyi et al., 2019) [unverified]
Study basis: discipline: MICROBIOLOGY; design: Investigation; population: 158 day-old and six 14-week old runted/stunted broiler chickens in commercial farms (ten) and hatcheries (six) located in Southwest Nigeria; focus: Southwest Nigeria commercial broiler enteric virus survey
```

Raw OpenAlex: `Adebiyi et al., 2019`  
Selected citation: `Adebiyi et al., 2019`  
Status/source: `VERIFIED_DOCUMENT` / `OPENALEX+DOCUMENT`

### Sample 3 — OPEN / UNVERIFIED

**Question**

Why was paraffin dissolved in nitrocellulose thinner before adding black extract in formulations B to E?

**Assistant target**

```text
In formulations B to E, the procedure begins by dissolving paraffin in nitrocellulose thinner, then adds the black extract dropwise with continuous stirring. Dissolving the paraffin in the vehicle before adding the extract promotes a more homogeneous mixture and helps avoid adding solid wax lumps. The dropwise addition with stirring further encourages uniform mixing.

Answer: In formulations B to E, paraffin was dissolved in nitrocellulose thinner.

Provenance: UNVERIFIED_STUDY
Citation: (YAKATUN & Alfa, 2025) [unverified]
Study basis: discipline: CHEMICAL_ENGINEERING; design: five different formulations; population: black plum leaves; location: Bida, Niger State, Nigeria; focus: Black plum printing ink production
```

Raw OpenAlex: `YAKATUN & Alfa, 2025`  
Selected citation: `YAKATUN & Alfa, 2025`  
Status/source: `METADATA_ONLY` / `OPENALEX+DOCUMENT`

### Sample 4 — OPEN / UNVERIFIED

**Question**

How is chromium singled out as the main contributor to child carcinogenic risk?

**Assistant target**

```text
For Cr, child LCR values exceeded the acceptable TLCR upper value of 10^-4 in every school except F, whose value 9.67E-05 was just below the cut-off; this makes Cr the sensitive carcinogenic endpoint.

Answer: Carcinogenic risks that exceed the Total life cancer risk (TLCR) acceptable values (10 -4 ) [18] were found in children exposed to Cr in all the schools except for F (9.67E-05).

Provenance: UNVERIFIED_STUDY
Citation: (Victor et al., 2018) [unverified]
Study basis: discipline: ENVIRONMENTAL_SCIENCE; design: Health risk assessment; population: children and adult; location: Rivers State, Nigeria; Obio/Akpor, Eleme and Ikwerre Local Government Areas; focus: Rivers State classroom corner dusts
```

Raw OpenAlex: `Victor et al., 2018`  
Selected citation: `Victor et al., 2018`  
Status/source: `VERIFIED_DOCUMENT` / `OPENALEX+DOCUMENT`

### Sample 5 — OPEN / UNVERIFIED

**Question**

At what temperature was the characteristic smell of ester recorded during preparation of Formulation A?

**Assistant target**

```text
20 o C

Provenance: UNVERIFIED_STUDY
Citation: (YAKATUN & Alfa, 2025) [unverified]
Study basis: discipline: CHEMICAL_ENGINEERING; design: five different formulations; population: black plum leaves; location: Bida, Niger State, Nigeria; focus: Black plum printing ink production
```

Raw OpenAlex: `YAKATUN & Alfa, 2025`  
Selected citation: `YAKATUN & Alfa, 2025`  
Status/source: `METADATA_ONLY` / `OPENALEX+DOCUMENT`

These are audit previews only; no Parquet generation was published.


In [ ]:
# ======================= stats + bounded table samples =======================
import json
import re

from IPython.display import Markdown, display

SAMPLE_ROWS = 6
TEXT_PREVIEW_CHARS = 220
SAMPLE_SEED = 7

TABLE_VIEWS = {
    "sft_examples": {
        "label": "Strict verified SFT",
        "groups": ["split", "mode", "support_route"],
        "columns": ["example_id", "paper_id", "split", "pair_type",
                    "mode", "support_route", "question", "response",
                    "citation_label", "citation_status",
                    "citation_metadata_source", "evidence_json", "token_estimate"],
    },
    "sft_mixed": {
        "label": "Mixed SFT (verified + support-unverified)",
        "groups": ["split", "verification_tier", "mode"],
        "columns": ["example_id", "paper_id", "split", "pair_type",
                    "mode", "verification_tier", "inclusion_source",
                    "reason_code", "question", "response",
                    "citation_raw_label", "citation_label",
                    "citation_status", "citation_metadata_source",
                    "evidence_json", "paper_context", "token_estimate"],
    },
    "dpo_pairs": {
        "label": "Strict verified preference / DPO",
        "groups": ["split", "support_route"],
        "columns": ["pair_id", "paper_id", "split", "support_route",
                    "question", "chosen", "rejected", "rejection_reason",
                    "citation_label", "citation_status",
                    "evidence_json", "token_estimate"],
    },
    "preference_mixed": {
        "label": "Mixed preference / DPO",
        "groups": ["split", "verification_tier", "support_route"],
        "columns": ["pair_id", "paper_id", "split", "verification_tier",
                    "inclusion_source", "reason_code", "question", "chosen",
                    "rejected", "rejection_reason", "citation_raw_label",
                    "citation_label", "citation_status", "evidence_json"],
    },
    "reranker_mixed": {
        "label": "Mixed reranker pairs",
        "groups": ["split", "verification_tier"],
        "columns": ["pair_id", "paper_id", "split", "question",
                    "positive_quote", "hard_negative_quote", "negative_reason",
                    "paper_context", "token_estimate"],
    },
    "quarantine": {
        "label": "Quarantine audit",
        "groups": ["stage", "reason_code", "pair_type"],
        "columns": ["record_id", "pair_id", "paper_id", "split",
                    "pair_type", "stage", "reason_code", "reason_detail",
                    "question", "target", "evidence_json"],
    },
    "discarded": {
        "label": "Hard rejects / discarded audit",
        "groups": ["stage", "reason_code", "pair_type"],
        "columns": ["record_id", "pair_id", "paper_id", "split",
                    "pair_type", "stage", "reason_code", "reason_detail",
                    "question", "target"],
    },
}

def _preview(value, limit=TEXT_PREVIEW_CHARS):
    if isinstance(value, (list, dict, tuple)):
        value = json.dumps(value, ensure_ascii=False)
    if pd.isna(value):
        return ""
    text = re.sub(r"\s+", " ", str(value)).strip()
    return text if len(text) <= limit else text[:limit - 1].rstrip() + "…"

def _show_output(name, spec):
    frame = OUTCOME.frames[name]
    display(Markdown(f"## `{name}.parquet` — {spec['label']}"))
    display(pd.DataFrame([{
        "rows": len(frame),
        "columns": len(frame.columns),
        "memory (MiB)": round(frame.memory_usage(deep=True).sum() / 2**20, 2),
    }]))

    group_columns = [column for column in spec["groups"] if column in frame.columns]
    display(Markdown("**Counts by the most useful routing fields**"))
    if len(frame) and group_columns:
        counts = (frame.groupby(group_columns, dropna=False).size()
                  .sort_values(ascending=False).head(30)
                  .rename("rows").reset_index())
        display(counts)
    else:
        display(pd.DataFrame({"status": ["No rows in this output"]}))

    display(Markdown(f"**Deterministic sample (up to {SAMPLE_ROWS} rows; long text shortened for display only)**"))
    columns = [column for column in spec["columns"] if column in frame.columns]
    if len(frame):
        sample = frame[columns].sample(min(SAMPLE_ROWS, len(frame)), random_state=SAMPLE_SEED)
        sample = sample.reset_index(drop=True).copy()
        for column in sample.columns:
            if sample[column].dtype == "object":
                sample[column] = sample[column].map(_preview)
        display(sample)
    else:
        display(pd.DataFrame(columns=columns))

print("Inspection helpers ready. Run the next seven cells for one output table per cell.")


Inspection helpers ready. Run the next seven cells for one output table per cell.


In [ ]:
# sft_examples.parquet: strict verified SFT
_show_output("sft_examples", TABLE_VIEWS["sft_examples"])


## `sft_examples.parquet` — Strict verified SFT

,rows,columns,memory (MiB)
0,168,34,3.16


**Counts by the most useful routing fields**

,split,mode,support_route,rows
0,train,CLOSED,OPEN_AS_IS,47
1,train,OPEN,OPEN_AS_IS,39
2,train,CLOSED,OPEN_WIDENED,38
3,train,OPEN,OPEN_WIDENED,24
4,evaluate,CLOSED,OPEN_WIDENED,6
5,evaluate,OPEN,OPEN_WIDENED,6
6,evaluate,CLOSED,OPEN_AS_IS,6
7,evaluate,OPEN,OPEN_AS_IS,2


**Deterministic sample (up to 6 rows; long text shortened for display only)**

,example_id,paper_id,split,pair_type,mode,support_route,question,response,citation_label,citation_status,citation_metadata_source,evidence_json,token_estimate
0,W2958945597:reasoning:reasoning-007:closed,W2958945597,train,REASONING,CLOSED,OPEN_WIDENED,Why is the absence of ANV in all day-old chick...,ANV causes baby chick nephropathy and was expe...,"Adebiyi et al., 2019",VERIFIED_DOCUMENT,OPENALEX+DOCUMENT,"[{""char_end"":34339,""char_start"":32616,""page"":9...",473
1,W4411511743:factual:F4:closed,W4411511743,train,FACTUAL,CLOSED,OPEN_AS_IS,Where was the raw black plum leaf material obt...,The raw material (black plum leaves) was obtai...,"YAKATUN & Alfa, 2025",METADATA_ONLY,OPENALEX+DOCUMENT,"[{""char_end"":6917,""char_start"":6516,""evidence_...",477
2,W2895189904:factual:F9:closed,W2895189904,train,FACTUAL,CLOSED,OPEN_WIDENED,Which instrument was used to analyze the soil ...,A gas chromatograph (GC) (Agilent 7890) equipp...,"Ouabo et al., 2018",VERIFIED_DOCUMENT,OPENALEX+DOCUMENT,"[{""char_end"":9059,""char_start"":8008,""page"":2,""...",348
3,W2897650568:factual:fact_014:closed,W2897650568,evaluate,FACTUAL,CLOSED,OPEN_AS_IS,Which exposure pathway was identified as the m...,oral ingestion followed by dermal contact and ...,"Victor et al., 2018",VERIFIED_DOCUMENT,OPENALEX+DOCUMENT,"[{""char_end"":21755,""char_start"":21632,""evidenc...",363
4,W3107928999:factual:F014:open,W3107928999,train,FACTUAL,OPEN,OPEN_AS_IS,What daily temperature is reported for the Olu...,25˚C Provenance: PROVIDED_EVIDENCE — Evidence ...,"Buba et al., 2020",METADATA_ONLY,OPENALEX+DOCUMENT,"[{""char_end"":16311,""char_start"":16026,""page"":5...",433
5,W4411511743:factual:F18:closed,W4411511743,train,FACTUAL,CLOSED,OPEN_WIDENED,How is the black plum tree (Vitex doniana) des...,The plant commonly known as black plum is a de...,"YAKATUN & Alfa, 2025",METADATA_ONLY,OPENALEX+DOCUMENT,"[{""char_end"":5447,""char_start"":3735,""page"":2,""...",421


In [ ]:
# sft_mixed.parquet: verified and support-unverified SFT
_show_output("sft_mixed", TABLE_VIEWS["sft_mixed"])


## `sft_mixed.parquet` — Mixed SFT (verified + support-unverified)

,rows,columns,memory (MiB)
0,409,34,10.25


**Counts by the most useful routing fields**

,split,verification_tier,mode,rows
0,train,UNVERIFIED,OPEN,220
1,train,VERIFIED,CLOSED,85
2,train,VERIFIED,OPEN,63
3,evaluate,UNVERIFIED,OPEN,21
4,evaluate,VERIFIED,CLOSED,12
5,evaluate,VERIFIED,OPEN,8


**Deterministic sample (up to 6 rows; long text shortened for display only)**

,example_id,paper_id,split,pair_type,mode,verification_tier,inclusion_source,reason_code,question,response,citation_raw_label,citation_label,citation_status,citation_metadata_source,evidence_json,paper_context,token_estimate
0,W2895189904:factual:F8:closed,W2895189904,train,FACTUAL,CLOSED,VERIFIED,STRICT_SFT,,What solvent and volume were used to elute the...,20 mL of n hexane. Provenance: LEARNED_STUDY C...,"Ouabo et al., 2018","Ouabo et al., 2018",VERIFIED_DOCUMENT,OPENALEX+DOCUMENT,"[{""char_end"":7980,""char_start"":7282,""page"":2,""...",Study context (scope metadata; factual support...,339
1,W2958945597:factual:factual-013:open,W2958945597,train,FACTUAL,OPEN,UNVERIFIED,SUPPORT_QUARANTINE,QUARANTINE_UNVERIFIED,Which GenBank accession numbers were assigned ...,"MK509014 (NGR_CAstV_Ch1), MK509015 (NGR_CAstV_...","Adebiyi et al., 2019","Adebiyi et al., 2019",VERIFIED_DOCUMENT,OPENALEX+DOCUMENT,"[{""char_end"":27898,""char_start"":27003,""page"":7...",Study context (scope metadata; factual support...,701
2,W4411511743:reasoning:R14:open,W4411511743,train,REASONING,OPEN,UNVERIFIED,SUPPORT_QUARANTINE,QUARANTINE_UNVERIFIED,Why was paraffin dissolved in nitrocellulose t...,"In formulations B to E, the procedure begins b...","YAKATUN & Alfa, 2025","YAKATUN & Alfa, 2025",METADATA_ONLY,OPENALEX+DOCUMENT,"[{""char_end"":11303,""char_start"":10445,""page"":6...",Study context (scope metadata; factual support...,739
3,W2897650568:reasoning:reason_017:open,W2897650568,evaluate,REASONING,OPEN,UNVERIFIED,SUPPORT_QUARANTINE,QUARANTINE_UNVERIFIED,How is chromium singled out as the main contri...,"For Cr, child LCR values exceeded the acceptab...","Victor et al., 2018","Victor et al., 2018",VERIFIED_DOCUMENT,OPENALEX+DOCUMENT,"[{""char_end"":33727,""char_start"":33562,""page"":8...",Study context (scope metadata; factual support...,664
4,W4411511743:factual:F6:open,W4411511743,train,FACTUAL,OPEN,UNVERIFIED,SUPPORT_QUARANTINE,QUARANTINE_UNVERIFIED,At what temperature was the characteristic sme...,20 o C Provenance: UNVERIFIED_STUDY Citation: ...,"YAKATUN & Alfa, 2025","YAKATUN & Alfa, 2025",METADATA_ONLY,OPENALEX+DOCUMENT,"[{""char_end"":10268,""char_start"":9531,""page"":5,...",Study context (scope metadata; factual support...,697
5,W3107928999:reasoning:R018:open,W3107928999,train,REASONING,OPEN,UNVERIFIED,SUPPORT_QUARANTINE,QUARANTINE_UNVERIFIED,How large was the dense forest recovery betwee...,The paper directly reports the 2014 dense fore...,"Buba et al., 2020","Buba et al., 2020",METADATA_ONLY,OPENALEX+DOCUMENT,"[{""char_end"":32040,""char_start"":30274,""page"":1...",Study context (scope metadata; factual support...,989


In [ ]:
# dpo_pairs.parquet: strict verified preference data
_show_output("dpo_pairs", TABLE_VIEWS["dpo_pairs"])


## `dpo_pairs.parquet` — Strict verified preference / DPO

,rows,columns,memory (MiB)
0,30,30,0.47


**Counts by the most useful routing fields**

,split,support_route,rows
0,train,OPEN_WIDENED,13
1,train,OPEN_AS_IS,13
2,evaluate,OPEN_WIDENED,3
3,evaluate,OPEN_AS_IS,1


**Deterministic sample (up to 6 rows; long text shortened for display only)**

,pair_id,paper_id,split,support_route,question,chosen,rejected,rejection_reason,citation_label,citation_status,evidence_json,token_estimate
0,W108716228:preference:p2,W108716228,train,OPEN_WIDENED,What conditions accompanied the reported incap...,On the frequency of incapacitation associated ...,45% of respondents were incapacitated exactly ...,MISSING_CONDITIONS,"Sam-Wobo et al., 2013",CORRECTED_DOCUMENT,"[{""char_end"":13837,""char_start"":12943,""page"":4...",830
1,W2895189904:preference:P5,W2895189904,train,OPEN_AS_IS,How long was the soil extract sonicated in the...,10 min. Provenance: PROVIDED_EVIDENCE — Eviden...,30 min. Provenance: PROVIDED_EVIDENCE — Eviden...,UNGROUNDED_NUMBER,"Ouabo et al., 2018",VERIFIED_DOCUMENT,"[{""char_end"":7439,""char_start"":7382,""evidence_...",431
2,W108716228:preference:p1,W108716228,train,OPEN_WIDENED,How certain should one be that the low prevale...,"Though, the prevalence of helminthiasis is low...",The low prevalence definitively represents the...,OVERCLAIM,"Sam-Wobo et al., 2013",CORRECTED_DOCUMENT,"[{""char_end"":17928,""char_start"":16251,""page"":6...",874
3,W1992790258:preference:P05,W1992790258,train,OPEN_WIDENED,How many dog vaccinations were recorded in Lag...,There was a yearly increase in the number of v...,A total of 2033 dogs were vaccinated in Lagos ...,MISSING_CONDITIONS,"Hambolu et al., 2013",CONFLICT,"[{""char_end"":19914,""char_start"":19275,""page"":5...",654
4,W2461070053:preference:P2,W2461070053,train,OPEN_WIDENED,What was the oil absorption of the Aro-Ndizuog...,32g/100g clay. Provenance: PROVIDED_EVIDENCE —...,32 g per 100 g resin. Provenance: PROVIDED_EVI...,WRONG_UNIT,"Igwe et al., 2016",VERIFIED_DOCUMENT,"[{""char_end"":13923,""char_start"":12640,""page"":4...",561
5,W1992790258:preference:P01,W1992790258,train,OPEN_AS_IS,What proportion of the study’s dog brain sampl...,Only 7 (1.58%) out of the 444 brain samples co...,This study found rabies antigen in 28–32% of a...,WRONG_ATTRIBUTION,"Hambolu et al., 2013",CONFLICT,"[{""char_end"":2608,""char_start"":2420,""evidence_...",524


In [ ]:
# preference_mixed.parquet: all structurally valid preference data
_show_output("preference_mixed", TABLE_VIEWS["preference_mixed"])


## `preference_mixed.parquet` — Mixed preference / DPO

,rows,columns,memory (MiB)
0,50,30,0.93


**Counts by the most useful routing fields**

,split,verification_tier,support_route,rows
0,train,UNVERIFIED,QUARANTINE_UNVERIFIED,19
1,train,VERIFIED,OPEN_AS_IS,13
2,train,VERIFIED,OPEN_WIDENED,13
3,evaluate,VERIFIED,OPEN_WIDENED,3
4,evaluate,VERIFIED,OPEN_AS_IS,1
5,evaluate,UNVERIFIED,QUARANTINE_UNVERIFIED,1


**Deterministic sample (up to 6 rows; long text shortened for display only)**

,pair_id,paper_id,split,verification_tier,inclusion_source,reason_code,question,chosen,rejected,rejection_reason,citation_raw_label,citation_label,citation_status,evidence_json
0,W2054540843:preference:PR04,W2054540843,train,UNVERIFIED,SUPPORT_QUARANTINE,PREFERENCE_UNVERIFIED,Could the plant extracts completely reverse si...,"No, because the authors state that these extra...","Yes, the extracts fully normalized sickle cell...",IGNORES_LIMITATION,"Afolabi et al., 2012","Afolabi et al., 2012",VERIFIED_DOCUMENT,"[{""char_end"":30054,""char_start"":28958,""page"":7..."
1,W2150275902:preference:P1,W2150275902,train,UNVERIFIED,SUPPORT_QUARANTINE,PREFERENCE_UNVERIFIED,What was the extract's inhibition at 400 mg/kg...,Table 1 lists 80.00% inhibition for the 400 mg...,The extract produced 100% suppression of paras...,UNGROUNDED_NUMBER,"Collins et al., 2014","Collins et al., 2014",VERIFIED_DOCUMENT,"[{""char_end"":3241,""char_start"":1445,""page"":1,""..."
2,W2461070053:preference:P3,W2461070053,train,VERIFIED,STRICT_DPO,,What is the pigment-to-binder ratio of the pai...,0.3 : 1.0. Provenance: PROVIDED_EVIDENCE — Evi...,0.5 : 1.0. Provenance: PROVIDED_EVIDENCE — Evi...,UNGROUNDED_NUMBER,"Igwe et al., 2016","Igwe et al., 2016",VERIFIED_DOCUMENT,"[{""char_end"":10640,""char_start"":10409,""evidenc..."
3,W3107928999:preference:P003,W3107928999,train,UNVERIFIED,SUPPORT_QUARANTINE,PREFERENCE_UNVERIFIED,Does this study prove that forest degradation ...,"No. It found a positive correlation of 0.58, b...",Yes. The positive correlation proves that fore...,CAUSAL_OVERREACH,"Buba et al., 2020","Buba et al., 2020",METADATA_ONLY,"[{""char_end"":36323,""char_start"":35947,""page"":1..."
4,W2897650568:preference:pref_001,W2897650568,evaluate,UNVERIFIED,SUPPORT_QUARANTINE,PREFERENCE_UNVERIFIED,Does this study prove that classroom dust caus...,Carcinogenic risks that exceed the Total life ...,"Yes, the study proves that classroom dust caus...",CAUSAL_OVERREACH,"Victor et al., 2018","Victor et al., 2018",VERIFIED_DOCUMENT,"[{""char_end"":33727,""char_start"":33562,""page"":8..."
5,W2895189904:preference:P3,W2895189904,train,UNVERIFIED,SUPPORT_QUARANTINE,PREFERENCE_UNVERIFIED,Which site had the highest ingestion cancer ri...,"New Bell, 1.9945E-07. Provenance: UNVERIFIED_S...","Ngodi, 1.9945E-07. Provenance: UNVERIFIED_STUD...",WRONG_ATTRIBUTION,"Ouabo et al., 2018","Ouabo et al., 2018",VERIFIED_DOCUMENT,"[{""char_end"":18890,""char_start"":18623,""page"":6..."


In [ ]:
# reranker_mixed.parquet: structurally valid reranker data
_show_output("reranker_mixed", TABLE_VIEWS["reranker_mixed"])


## `reranker_mixed.parquet` — Mixed reranker pairs

,rows,columns,memory (MiB)
0,50,19,0.4


**Counts by the most useful routing fields**

,split,verification_tier,rows
0,train,UNVERIFIED,45
1,evaluate,UNVERIFIED,5


**Deterministic sample (up to 6 rows; long text shortened for display only)**

,pair_id,paper_id,split,question,positive_quote,hard_negative_quote,negative_reason,paper_context,token_estimate
0,W2054540843:reranker:RR04,W2054540843,train,Why is catalase useful as a defense against ox...,The catalase enzyme is an endogenous antioxida...,Catalase activity is induced by the presence o...,SAME_MATERIAL_OTHER_PROPERTY,Study context (scope metadata; factual support...,92
1,W2150275902:reranker:RR1,W2150275902,train,What is the extract's suppressive antiplasmodi...,The methanol leaf extract of L. lanceolata exh...,"The methanol extract, at the same doses (100, ...",SAME_PAPER_OTHER_SECTION,Study context (scope metadata; factual support...,122
2,W2461070053:reranker:RR3,W2461070053,train,How did the formulated paint samples behave in...,The dry films of the formulated paint sample s...,The dry paint film sample containing 60 % loca...,SAME_MATERIAL_OTHER_PROPERTY,Study context (scope metadata; factual support...,65
3,W3107928999:reranker:RR003,W3107928999,train,Which satellite sensor acquired the 2014 imagery?,|Landsat 8|Operational Land Imager (OLI)|5 Jan...,|Landsat 7|Enhanced Thematic Mapper plus (ETM+...,SAME_PROPERTY_OTHER_MATERIAL,Study context (scope metadata; factual support...,52
4,W2897650568:reranker:rerank_001,W2897650568,evaluate,What is the highest chromium concentration rep...,while that of Cr and As were obtained from H a...,However Chromium levels obtained were below th...,SAME_PROPERTY_OTHER_MATERIAL,Study context (scope metadata; factual support...,109
5,W2895189904:reranker:RE3,W2895189904,train,What was the ingestion cancer risk at the thre...,|Ingestion|8.7945E-08|8.4932E-08|1.9945E-07|,The obtained TEQ values of dioxin-like PCBs in...,SAME_PAPER_OTHER_SECTION,Study context (scope metadata; factual support...,72


In [ ]:
# quarantine.parquet: unresolved support and reranker audit rows
_show_output("quarantine", TABLE_VIEWS["quarantine"])


## `quarantine.parquet` — Quarantine audit

,rows,columns,memory (MiB)
0,311,17,2.04


**Counts by the most useful routing fields**

,stage,reason_code,pair_type,rows
0,SUPPORT,QUARANTINE_UNVERIFIED,REASONING,184
1,SUPPORT,QUARANTINE_UNVERIFIED,FACTUAL,57
2,RERANKER,HARD_NEGATIVE_NOT_VALIDATED,RERANKER,50
3,SUPPORT,QUARANTINE_UNVERIFIED,PREFERENCE,20


**Deterministic sample (up to 6 rows; long text shortened for display only)**

,record_id,pair_id,paper_id,split,pair_type,stage,reason_code,reason_detail,question,target,evidence_json
0,W2461070053:factual:F14:SUPPORT,W2461070053:factual:F14,W2461070053,train,FACTUAL,SUPPORT,QUARANTINE_UNVERIFIED,one or more target sentences lack source support,What surface dry period did the prepared paint...,approximately 3 hours,"[{""char_end"":17157,""char_start"":16530,""page"":5..."
1,W2897650568:reasoning:reason_005:SUPPORT,W2897650568:reasoning:reason_005,W2897650568,evaluate,REASONING,SUPPORT,QUARANTINE_UNVERIFIED,one or more target sentences lack source support,Why are lead concentrations in classroom dust ...,Lead has a toxic effect on the central nervous...,"[{""char_end"":15597,""char_start"":15409,""page"":4..."
2,W3107928999:preference:P002:SUPPORT,W3107928999:preference:P002,W3107928999,train,PREFERENCE,SUPPORT,QUARANTINE_UNVERIFIED,figure is not bound to the asked entity,What is the annual rainfall of the Oluwa Fores...,The study area has an annual rainfall which ex...,"[{""char_end"":16564,""char_start"":15640,""page"":5..."
3,W2150275902:reasoning:R13:SUPPORT,W2150275902:reasoning:R13,W2150275902,train,REASONING,SUPPORT,QUARANTINE_UNVERIFIED,one or more target sentences lack source support,How does the suppressive result support the co...,The logical step is: the suppressive test show...,"[{""char_end"":22587,""char_start"":20873,""page"":7..."
4,W1992790258:reasoning:R10:SUPPORT,W1992790258:reasoning:R10,W1992790258,train,REASONING,SUPPORT,QUARANTINE_UNVERIFIED,evidence negates the positive target,Why were males more often bitten than females ...,The records show 92 male victims and 87 female...,"[{""char_end"":17173,""char_start"":16387,""page"":5..."
5,W2150275902:reasoning:R10:SUPPORT,W2150275902:reasoning:R10,W2150275902,train,REASONING,SUPPORT,QUARANTINE_UNVERIFIED,named term absent from evidence,Why is this study's focus on a plant from Nsuk...,The introduction establishes Nigeria's outsize...,"[{""char_end"":9222,""char_start"":8986,""page"":2,""..."


In [ ]:
# discarded.parquet: hard rejects and duplicate-conflict audit rows
_show_output("discarded", TABLE_VIEWS["discarded"])
print("Notebook previews are bounded; every Parquet retains all rows and full text.")


## `discarded.parquet` — Hard rejects / discarded audit

,rows,columns,memory (MiB)
0,0,13,0.0


**Counts by the most useful routing fields**

,status
0,No rows in this output


**Deterministic sample (up to 6 rows; long text shortened for display only)**

,record_id,pair_id,paper_id,split,pair_type,stage,reason_code,reason_detail,question,target


Notebook previews are bounded; every Parquet retains all rows and full text.
